# 03 · Branching, Merging & Rebasing

**Module 16 – Git & Version Control** · Estimated time: 55 min

> From 📘 *Branching & Merging*, with a real merge conflict resolved step by step, plus **rebase** (which the handbook only mentions as an "advanced" exercise).

### 🎯 What you will learn
- What a branch is (a movable pointer to a commit)
- Creating and switching branches: ✅ `git switch` (and ⚠️ the older `git checkout`)
- Fast-forward vs three-way merges
- Resolving merge conflicts; "ours" vs "theirs"
- Managing branches: delete, rename, list merged
- Branching strategies & naming conventions
- `git rebase` and the golden rule

In [1]:
import os, tempfile
PLAYGROUND = tempfile.mkdtemp(prefix="git-playground-")
os.environ.update({
    "PLAYGROUND": PLAYGROUND, "GIT_CONFIG_GLOBAL": os.devnull,
    "GIT_AUTHOR_NAME": "Demo Learner", "GIT_AUTHOR_EMAIL": "learner@example.com",
    "GIT_COMMITTER_NAME": "Demo Learner", "GIT_COMMITTER_EMAIL": "learner@example.com",
    "GIT_PAGER": "cat", "GIT_EDITOR": "true", "GIT_MERGE_AUTOEDIT": "no",
})

In [2]:
%%bash
cd "$PLAYGROUND" && git init -q -b main site && cd site
echo "<h1>Welcome</h1>" > index.html
git add . && git commit -q -m "Initial commit"
git log --oneline

51fbee1 Initial commit


## 1. Why branches matter

Branches let you:
- work on features **without affecting** the main code
- experiment safely (throw the branch away if it fails)
- collaborate without stepping on each other's toes
- maintain multiple versions (e.g. `release/2.0` and `main`)

Think of branches as **parallel universes** of your code. A branch is just a **movable pointer to a commit**; creating one is instant and cheap.

```
main:          A ── B ── C
                          \
feature-login:             D ── E
```

## 2. Viewing, creating and switching

In [3]:
%%bash
cd "$PLAYGROUND/site"
git branch                         # list local branches; * marks the current one
git branch feature-navbar          # create a branch (doesn't switch)
git switch feature-navbar          # ✅ switch (Git 2.23+)
git branch
git switch main
git switch -c feature-login        # ✅ create AND switch in one step
git branch -a                      # -a also lists remote branches

* main


Switched to branch 'feature-navbar'


* feature-navbar
  main


Switched to branch 'main'


Switched to a new branch 'feature-login'


* feature-login
  feature-navbar
  main


> ⚠️ **Older equivalents (📘):** `git checkout feature-navbar` (switch) and `git checkout -b feature-login` (create + switch). `checkout` still works but does too many unrelated things (switch branches *and* discard file changes), so Git split it into `git switch` and `git restore`.

## 3. Work on a branch

In [4]:
%%bash
cd "$PLAYGROUND/site"
git switch -q feature-login
echo "<form>Login form</form>" > login.html
git add login.html && git commit -q -m "Add login form"
echo "On feature-login:"; ls
git switch -q main
echo "On main (login.html doesn't exist here!):"; ls

On feature-login:


index.html
login.html


On main (login.html doesn't exist here!):


index.html


## 4. Merging

### Fast-forward merge
If `main` hasn't moved since the branch was created, Git just moves the `main` pointer forward: no merge commit.

In [5]:
%%bash
cd "$PLAYGROUND/site"
git switch -q main
git merge feature-login
git log --oneline --graph --all

Updating 51fbee1..b7d2686
Fast-forward


 login.html | 1 +
 1 file changed, 1 insertion(+)
 create mode 100644 login.html


* b7d2686 Add login form
* 51fbee1 Initial commit


### Three-way merge
If **both** branches have new commits, Git creates a **merge commit** with two parents.

In [6]:
%%bash
cd "$PLAYGROUND/site"
git switch -q feature-navbar
echo "<nav>Home | About</nav>" > navbar.html
git add . && git commit -q -m "Add navbar"
git switch -q main
echo "<footer>© 2026</footer>" > footer.html
git add . && git commit -q -m "Add footer"
git merge feature-navbar -m "Merge feature-navbar"
git log --oneline --graph --all

Merge made by the 'ort' strategy.


 navbar.html | 1 +
 1 file changed, 1 insertion(+)
 create mode 100644 navbar.html


*   3fc1773 Merge feature-navbar
|\  


| * e0af543 Add navbar


* | 1bcb768 Add footer
* | b7d2686 Add login form
|/  
* 51fbee1 Initial commit


## 5. Resolving merge conflicts

A **conflict** happens when both branches changed the **same lines** of the same file.

In [7]:
%%bash
cd "$PLAYGROUND/site"
git switch -q -c feature-branch
echo "<h1>Welcome to My Website</h1>" > index.html
git commit -q -am "Change heading on feature branch"
git switch -q main
echo "<h1>Welcome to Our Site</h1>" > index.html
git commit -q -am "Change heading on main"
git merge feature-branch || true          # '|| true' only so the notebook continues after the conflict
echo "------ git status ------"
git status --short

Auto-merging index.html
CONFLICT (content): Merge conflict in index.html


Automatic merge failed; fix conflicts and then commit the result.


------ git status ------


UU index.html


In [8]:
%%bash
cd "$PLAYGROUND/site"
cat index.html

<<<<<<< HEAD
<h1>Welcome to Our Site</h1>
<h1>Welcome to My Website</h1>
>>>>>>> feature-bra

nch


Conflict markers:
```
<<<<<<< HEAD               ← your current branch's version (main)
<h1>Welcome to Our Site</h1>
=======
<h1>Welcome to My Website</h1>
>>>>>>> feature-branch     ← the incoming branch's version
```

**To resolve:**
1. Open the conflicted file (VS Code shows *Accept Current / Incoming / Both* buttons)
2. Decide what the final content should be
3. **Remove the markers**
4. `git add` the file, then `git commit`

In [9]:
%%bash
cd "$PLAYGROUND/site"
echo "<h1>Welcome to Our Website</h1>" > index.html       # our manual resolution (a bit of both)
git add index.html
git commit -q -m "Resolve merge conflict in index.html"
git log --oneline --graph -6
cat index.html

*   d622cda Resolve merge conflict in index.html
|\  
| * 9dd8da7 Change heading on feature branch


* | 7814c57 Change heading on main
|/  


*   3fc1773 Merge feature-navbar
|\  
| * e0af543 Add navbar
* | 1bcb768 Add footer


<h1>Welcome to Our Website</h1>


### "Ours" vs "theirs" (📘)
During a merge:
- **ours** = the branch you're **on** (merging *into*)
- **theirs** = the branch you're merging **from** (incoming)

```bash
git checkout --ours index.html      # keep the current branch's version of the whole file
git checkout --theirs index.html    # take the incoming version of the whole file
# modern equivalents: git restore --ours / --theirs index.html
git add index.html && git commit
```
These only work **during an active conflict**, and they replace the **entire file**, not individual sections.

Other tools: `git mergetool` (opens a visual merge tool), and **`git merge --abort`** to cancel the merge and return to the state before it.

## 6. Managing branches

In [10]:
%%bash
cd "$PLAYGROUND/site"
echo "--- merged into main:"; git branch --merged
git switch -q -c experiment && echo "x" > x.txt && git add x.txt && git commit -q -m "Experiment" && git switch -q main
echo "--- not merged:"; git branch --no-merged
git branch -d feature-login             # delete a merged branch
git branch -D experiment                # force-delete an UNmerged branch (its commits are abandoned)
git branch -m feature-navbar feature/navbar    # rename a branch
git branch

--- merged into main:


  feature-branch
  feature-login
  feature-navbar
* main


--- not merged:


  experiment


Deleted branch feature-login (was b7d2686).


Deleted branch experiment (was 509ac7e).


  feature-branch
  feature/navbar
* main


### Comparing branches

In [11]:
%%bash
cd "$PLAYGROUND/site"
git switch -q -c feature/search
echo "search" > search.html && git add . && git commit -q -m "Add search page"
echo "--- commits in feature/search that are not in main:"
git log --oneline main..feature/search
git switch -q main

--- commits in feature/search that are not in main:


3a76911 Add search page


## 7. Rebasing (📘 "advanced" exercise, explained)

`git rebase main` takes your branch's commits and **replays them on top of** the latest `main`, producing a **linear history** without a merge commit.

```
Before:        A ── B ── C        (main)
                     \
                      D ── E      (feature)

After rebase:  A ── B ── C ── D' ── E'   (feature, rebuilt on top of C)
```

In [12]:
%%bash
cd "$PLAYGROUND/site"
echo "<p>news</p>" > news.html && git add . && git commit -q -m "Add news page on main"
git switch -q feature/search
git rebase main
git log --oneline --graph -4
git switch -q main && git merge -q feature/search     # now a fast-forward
git log --oneline -3

Successfully rebased and updated refs/heads/feature/search.


* c982f56 Add search page


* ccbb227 Add news page on main


*   d622cda Resolve merge conflict in index.html
|\  
| * 9dd8da7 Change heading on feature branch


c982f56 Add search page


ccbb227 Add news page on main
d622cda Resolve merge conflict in index.html


> ⚠️ **The golden rule of rebasing:** never rebase commits that you've already pushed and others may have based work on. Rebase rewrites commit IDs. Use it on your **local / personal** branches to tidy up before merging.
>
> Also useful: `git rebase -i HEAD~3` (interactive: squash, reorder, reword commits) and `git pull --rebase`.

## 8. Branching strategies (📘)

### Feature-branch workflow
```bash
git switch -c feature/shopping-cart     # 1. create a feature branch
# ... commit work ...                   # 2. work on the feature
git switch main && git merge feature/shopping-cart   # 3. merge back (usually via a Pull Request on GitHub)
git branch -d feature/shopping-cart     # 4. delete the branch
```

### Hotfix workflow
```bash
git switch main && git switch -c hotfix/security
git commit -am "Fix security vulnerability"
git switch main && git merge hotfix/security
git switch develop && git merge hotfix/security     # also merge into develop if you use one
```

Popular models: **GitHub Flow** (main + short-lived feature branches + PRs: simple and the most common), **trunk-based development**, and **Git Flow** (main/develop/release/hotfix: heavier, for scheduled releases).

### Naming conventions
| Prefix | Use |
|---|---|
| `feature/` | new features (`feature/user-login`) |
| `bugfix/` / `fix/` | bug fixes (`bugfix/header-alignment`) |
| `hotfix/` | urgent production fixes |
| `release/` | release preparation (`release/v2.0`) |
| `chore/` | maintenance (`chore/update-dependencies`) |

### Best practices
1. Keep branches focused: one feature per branch
2. Use descriptive names: `feature/user-auth`, not `new-stuff`
3. Delete merged branches
4. Merge (or rebase on) `main` regularly; don't let branches diverge too far
5. Test before merging (CI runs your tests on every PR)

## 9. Command summary (📘)

| Command | Description |
|---|---|
| `git branch` | list branches |
| `git branch <name>` | create a branch |
| `git switch <branch>` / ⚠️ `git checkout <branch>` | switch |
| `git switch -c <branch>` / ⚠️ `git checkout -b <branch>` | create + switch |
| `git merge <branch>` | merge into the current branch |
| `git merge --abort` | cancel a conflicted merge |
| `git rebase <branch>` | replay commits on top of another branch |
| `git branch -d / -D <branch>` | delete (safe / force) |
| `git branch -m <old> <new>` | rename |
| `git log --graph --oneline --all` | visualise branches |
| `git log main..feature` | commits in feature not in main |

## 🏋️ Practice (📘 exercise)
1. Create a repository with `main.txt` containing "Main branch content".
2. Create branch `feature-a`, add `feature-a.txt`, commit.
3. Switch back to `main`, create `feature-b`, add `feature-b.txt`, commit.
4. Merge `feature-a` into `main`, then `feature-b` (one fast-forward, one three-way merge: which is which?).
5. Create a conflict intentionally (edit the same line on two branches) and resolve it.
6. **Advanced:** redo the exercise using `git rebase` instead of merging, to keep a linear history.